# 🛢️ PumpGuard AI — Workshop IoT Predictive Maintenance
**v2.0** &nbsp;·&nbsp; Thời lượng: 3–4 giờ &nbsp;·&nbsp; Cấp độ: Trung cấp &nbsp;·&nbsp; Nền tảng: Google Colab

---

## 🎯 Mục tiêu

Xây dựng hệ thống IoT Predictive Maintenance thực tế từng bước, bao gồm:

| Layer | Công nghệ | Vai trò |
|-------|-----------|---------|
| Sensor | Python simulator | Giả lập 52 cảm biến công nghiệp |
| Transport | MQTT / HiveMQ Cloud | Truyền dữ liệu pub/sub |
| Pipeline | Node-RED / FlowFuse | Xử lý, phát hiện bất thường |
| Backend | FastAPI + WebSocket | API server, broadcast real-time |
| AI | Claude / GPT-4 / Gemini | Phân tích & khuyến nghị bảo trì |
| Alerting | Resend | Email cảnh báo tự động |
| Dashboard | HTML / JS / Chart.js | Giao diện trực tiếp trên trình duyệt |

---

## 🏗️ Kiến trúc

```
☁️  CLOUD (chính)                    💻  LOCAL FALLBACK

[Simulator] → [HiveMQ Cloud] → [FlowFuse/Node-RED]
[Simulator] → [Mosquitto:1883] → [Node-RED:1880]
                                         │ HTTP POST
                                 [FastAPI :8000 on Colab]
                                         │ Cloudflare Tunnel
                                   [Dashboard – Browser]
```

---

## 📋 Cách dùng notebook

> Mỗi phần = **lý thuyết → thực hành → xác nhận**.
> File source code do giảng viên cung cấp — bạn sẽ **upload từng file** vào Colab theo hướng dẫn.
> Cell `☁️ CLOUD` và `💻 FALLBACK` chọn **một trong hai**, không cần chạy cả hai.

---
# ⚙️ Chuẩn bị trước (Prerequisites)

Hoàn thành trước khi bắt đầu workshop:

---

## 1 · Tài khoản HiveMQ Cloud *(MQTT Broker)*

1. Truy cập **[console.hivemq.cloud](https://console.hivemq.cloud/)** → Đăng ký → **Serverless Free**
2. Tạo cluster → vào **Access Management** → tạo credential: `pumpguard / <password>`
3. Ghi lại:
```
HIVEMQ_HOST = xxxxxxxx.s1.eu.hivemq.cloud
HIVEMQ_USER = pumpguard
HIVEMQ_PASS = **********
```

---

## 2 · Tài khoản FlowFuse *(Node-RED hosting)*

1. Truy cập **[app.flowfuse.com](https://app.flowfuse.com/)** → Đăng ký → **Free tier**
2. Tạo **Application** → đặt tên `pumpguard-workshop`
3. Tạo **Instance** → chọn Node-RED 3.x → đợi khởi động
4. Ghi lại URL instance: `https://xxxxxxxx.flowfuse.cloud`

---

## 3 · File source code

Giải nén **`pump-iot-demo.zip`** (do giảng viên cung cấp) ra thư mục trên máy tính của bạn.
Bạn sẽ upload từng file theo hướng dẫn trong notebook.

```
pump-iot-demo/           ← giải nén ZIP ra đây trên máy tính
├── backend/
│   └── server.py        ← upload ở Phần 3
├── dashboard/
│   ├── index.html       ← upload ở Phần 6
│   └── control.html     ← upload ở Phần 6
├── nodered/
│   └── flows.json       ← upload ở Phần 5
└── scripts/
    └── simulate_sensor.py  ← upload ở Phần 7
```

---
# Phần 1 — Khởi động & Tạo Workspace

## 1.1 Kiểm tra Runtime

In [ ]:
import sys, platform, subprocess
print(f'Python  : {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
mem = subprocess.run(['free','-h'], capture_output=True, text=True).stdout
print('Memory:\n' + mem)
assert sys.version_info >= (3,8), 'Cần Python 3.8+'
print('✅ Runtime OK')

## 1.2 Tạo cấu trúc thư mục dự án

Chạy cell này để tạo sẵn các thư mục.
Bạn sẽ upload file vào đây theo từng phần trong notebook.

```
/content/pump-iot-demo/
├── backend/        ← server.py, .env         (Phần 3)
├── dashboard/      ← index.html, control.html (Phần 6)
├── nodered/        ← flows.json               (Phần 5)
└── scripts/        ← simulate_sensor.py       (Phần 7)
```

In [ ]:
import os
PROJ = '/content/pump-iot-demo'
for d in ['backend', 'dashboard', 'nodered', 'scripts']:
    os.makedirs(os.path.join(PROJ, d), exist_ok=True)
    print(f'  📁 {PROJ}/{d}/')
print('\n✅ Cấu trúc thư mục đã tạo. Sẵn sàng nhận file!')

---
# Phần 2 — Cài đặt Dependencies

## 2.1 Python packages

| Package | Vai trò |
|---------|---------|
| `fastapi` + `uvicorn` | Web framework + ASGI server |
| `paho-mqtt` | MQTT client để subscribe từ broker |
| `python-dotenv` | Đọc biến từ file `.env` |
| `anthropic` / `openai` / `google-generativeai` | AI SDKs |
| `resend` | Email API |
| `pycloudflared` | Tạo public URL cho Colab |

In [ ]:
print('📥 Cài đặt Python packages...')
!pip install -q fastapi 'uvicorn[standard]' paho-mqtt python-dotenv 2>&1 | tail -3
!pip install -q anthropic openai google-generativeai resend pycloudflared 2>&1 | tail -3

import importlib
ok = True
for pkg in ['fastapi','uvicorn','paho','anthropic','resend']:
    try:
        importlib.import_module(pkg)
        print(f'  ✅ {pkg}')
    except ImportError:
        print(f'  ❌ {pkg}')
        ok = False
print('\n✅ Xong!' if ok else '\n⚠️ Thử chạy lại.')

## 2.2 💻 FALLBACK — Cài Mosquitto & Node-RED *(Bỏ qua nếu dùng HiveMQ + FlowFuse)*

In [ ]:
# ── 💻 FALLBACK ONLY ─────────────────────────────────────────
print('📥 Cài Mosquitto...')
!apt-get install -y -q mosquitto mosquitto-clients 2>&1 | tail -3
print('📥 Cài Node.js 20...')
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>&1 | tail -3
!apt-get install -y -q nodejs 2>&1 | tail -3
print('📥 Cài Node-RED...')
!npm install -g --quiet node-red 2>&1 | tail -3
!node-red --version
print('✅ Mosquitto + Node-RED sẵn sàng')

---
# Phần 3 — Backend: `server.py`

## server.py làm gì?

File này là trung tâm xử lý của hệ thống:

```
            ┌──────────────────────────────────────┐
            │           FastAPI Server              │
Node-RED ──►│ POST /sensor-update → broadcast WS   │──► Dashboard
Dashboard ──│ WS /ws ← WebSocket pool               │
Node-RED ──►│ POST /alert → AI + Email              │
Browser ───►│ GET /dashboard/ → index.html          │
            └──────────────────────────────────────┘
```

**Các thành phần chính:**
- `ConnectionManager` — quản lý pool WebSocket clients
- `POST /sensor-update` — nhận data từ Node-RED, broadcast real-time tới dashboard
- `POST /alert` — throttle 60s, gọi AI API, gửi email
- `StaticFiles` — serve `index.html` / `control.html`

### 📋 Thực hành: Upload `server.py`

*File backend chính — cần có trước khi khởi động FastAPI.*

1. Mở **Files panel** bên trái *(icon 📁 hoặc `Ctrl+Shift+F`)*
2. Điều hướng đến thư mục: **`/content/pump-iot-demo/backend/`**
3. Kéo thả file **`server.py`** vào &nbsp;*hoặc* chuột phải → **Upload**

| &nbsp; | Đường dẫn |
|--------|-----------|
| 📦 Trong ZIP | `pump-iot-demo/backend/server.py` |
| 📍 Upload đến | `/content/pump-iot-demo/backend/server.py` |

> Sau khi upload xong, chạy cell bên dưới để xác nhận.

In [ ]:
import os
_p = '/content/pump-iot-demo/backend/server.py'
assert os.path.exists(_p), '❌ Chưa thấy server.py — hãy upload lại'
_kb = max(1, os.path.getsize(_p) // 1024)
print('✅ server.py (' + str(_kb) + ' KB) — đã sẵn sàng!')
print('\n--- 12 dòng đầu ---')
with open(_p, encoding='utf-8', errors='replace') as _f:
    for _i, _l in enumerate(_f):
        if _i >= 12: print('  ...'); break
        print(f'  {_i+1:3d}  {_l.rstrip()}')

## 3.2 Tạo file `.env`

File `.env` chứa API keys — **tạo bằng code** vì cần điền thông tin cá nhân.
**Không bao giờ** chia sẻ hoặc commit file này lên Git.

Điền thông tin của bạn vào các biến bên dưới, sau đó chạy cell:

In [ ]:
import os
# ════════════════════════════════════════
# ✏️ Điền thông tin của bạn
# ════════════════════════════════════════
HIVEMQ_HOST = 'YOUR_CLUSTER.s1.eu.hivemq.cloud'
HIVEMQ_USER = 'pumpguard'
HIVEMQ_PASS = 'YOUR_PASSWORD'
HIVEMQ_PORT = 8883

AI_PROVIDER   = 'anthropic'   # 'anthropic' | 'openai' | 'gemini'
ANTHROPIC_KEY = ''
OPENAI_KEY    = ''
GEMINI_KEY    = ''

RESEND_KEY  = ''
ALERT_FROM  = 'alerts@yourdomain.com'
ALERT_TO    = 'your@email.com'
# ════════════════════════════════════════

env_lines = [
    '# PumpGuard AI — Environment Config',
    f'MQTT_HOST={HIVEMQ_HOST}',
    f'MQTT_PORT={HIVEMQ_PORT}',
    f'MQTT_USERNAME={HIVEMQ_USER}',
    f'MQTT_PASSWORD={HIVEMQ_PASS}',
    'MQTT_TLS=true',
    f'AI_PROVIDER={AI_PROVIDER}',
    f'ANTHROPIC_API_KEY={ANTHROPIC_KEY}',
    f'OPENAI_API_KEY={OPENAI_KEY}',
    f'GEMINI_API_KEY={GEMINI_KEY}',
    f'RESEND_API_KEY={RESEND_KEY}',
    f'ALERT_FROM={ALERT_FROM}',
    f'ALERT_TO={ALERT_TO}',
]
env_path = '/content/pump-iot-demo/backend/.env'
with open(env_path, 'w') as f:
    f.write('\n'.join(env_lines))
print('✅ .env đã tạo: ' + env_path)
print('\nNội dung (values ẩn):')
with open(env_path) as f:
    for line in f:
        if '=' in line and not line.startswith('#'):
            k, v = line.strip().split('=', 1)
            masked = (v[:3]+'***') if len(v) > 4 else ('(trống)' if not v else v)
            print(f'  {k:25s} = {masked}')

---
# Phần 4 — MQTT Broker

## MQTT là gì?

**MQTT** là giao thức pub/sub nhẹ, chuẩn công nghiệp cho IoT:

```
 Simulator (Publisher) ──publish──► MQTT Broker ──subscribe──► Node-RED
                              topic: pump/sensors
```

Dự án dùng **HiveMQ Cloud** — broker MQTT trên cloud, miễn phí, không cài đặt.

---
## ☁️ CLOUD — Kiểm tra kết nối HiveMQ

In [ ]:
import paho.mqtt.client as mqtt, ssl, os, threading
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

HOST = os.environ.get('MQTT_HOST', '')
PORT = int(os.environ.get('MQTT_PORT', 8883))
USER = os.environ.get('MQTT_USERNAME', '')
PASS = os.environ.get('MQTT_PASSWORD', '')

if not HOST or 'YOUR_CLUSTER' in HOST:
    print('⚠️ Chưa cấu hình MQTT_HOST — cập nhật .env ở Phần 3 trước')
else:
    _ok = threading.Event()
    def _on_connect(c, ud, f, rc, p=None):
        if rc == 0: _ok.set()
    c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, 'ws-test')
    c.username_pw_set(USER, PASS)
    c.tls_set(tls_version=ssl.PROTOCOL_TLS_CLIENT)
    c.on_connect = _on_connect
    c.connect_async(HOST, PORT)
    c.loop_start()
    result = _ok.wait(timeout=10)
    c.loop_stop(); c.disconnect()
    if result:
        print('✅ HiveMQ Cloud kết nối thành công!')
        print('   ' + HOST + ':' + str(PORT) + ' (TLS)')
    else:
        print('❌ Kết nối thất bại — kiểm tra host/user/pass trong .env')

---
## 💻 FALLBACK — Khởi động Mosquitto local *(Bỏ qua nếu dùng HiveMQ Cloud)*

In [ ]:
# ── 💻 FALLBACK ONLY ─────────────────────────────────────────
import subprocess, time, os
os.makedirs('/etc/mosquitto/conf.d', exist_ok=True)
with open('/etc/mosquitto/conf.d/workshop.conf', 'w') as f:
    f.write('listener 1883\nallow_anonymous true\n')
mosquitto_proc = subprocess.Popen(
    ['mosquitto', '-c', '/etc/mosquitto/conf.d/workshop.conf'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
if mosquitto_proc.poll() is None:
    print(f'✅ Mosquitto | localhost:1883 (PID {mosquitto_proc.pid})')
else:
    print('❌ Mosquitto lỗi — chạy: !mosquitto -v')

---
# Phần 5 — Node-RED Pipeline

## Pipeline xử lý dữ liệu sensor

```
[MQTT In: pump/sensors]
       │
[Parse & Validate]      ← kiểm tra schema, loại bỏ NaN
       │
[Rolling Buffer 60]     ← giữ 60 readings (~30s) trong RAM
       │
[Compute Trends]        ← avg, slope, std_dev, anomaly_score
       │
       ├─(score < 0.4)──► [POST /sensor-update]  ← update dashboard
       └─(score ≥ 0.4)──► [Throttle 60s] ──► [POST /alert]  ← AI + email
```

## ☁️ CLOUD — Upload `flows.json` + Import vào FlowFuse

### 📋 Thực hành: Upload `flows.json`

*File cấu hình Node-RED pipeline — cần upload trước khi generate flow cho FlowFuse.*

1. Mở **Files panel** bên trái *(icon 📁 hoặc `Ctrl+Shift+F`)*
2. Điều hướng đến thư mục: **`/content/pump-iot-demo/nodered/`**
3. Kéo thả file **`flows.json`** vào &nbsp;*hoặc* chuột phải → **Upload**

| &nbsp; | Đường dẫn |
|--------|-----------|
| 📦 Trong ZIP | `pump-iot-demo/nodered/flows.json` |
| 📍 Upload đến | `/content/pump-iot-demo/nodered/flows.json` |

> Sau khi upload xong, chạy cell bên dưới để xác nhận.

In [ ]:
import os
_p = '/content/pump-iot-demo/nodered/flows.json'
assert os.path.exists(_p), '❌ Chưa thấy flows.json — hãy upload lại'
_kb = max(1, os.path.getsize(_p) // 1024)
print('✅ flows.json (' + str(_kb) + ' KB) — đã sẵn sàng!')

### Tạo `flow_cloud.json` với HiveMQ credentials

Cell bên dưới đọc `flows.json` gốc và tự động điền HiveMQ credentials vào.
Kết quả là file `flow_cloud.json` — bạn sẽ download và import vào FlowFuse.

In [ ]:
import json, os
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

HOST = os.environ.get('MQTT_HOST', 'localhost')
PORT = int(os.environ.get('MQTT_PORT', 8883))
USER = os.environ.get('MQTT_USERNAME', '')
PASS = os.environ.get('MQTT_PASSWORD', '')

# PUBLIC_URL sẽ có sau Phần 6 — dùng placeholder cho bây giờ
_raw = globals().get('PUBLIC_URL', 'REPLACE_AFTER_TUNNEL')
PUBLIC_URL = _raw.tunnel if hasattr(_raw, 'tunnel') else str(_raw)

flow_src = '/content/pump-iot-demo/nodered/flows.json'
with open(flow_src) as f:
    flow = json.load(f)

nodes = flow if isinstance(flow, list) else flow.get('flows', [])
for node in nodes:
    if node.get('type') == 'mqtt-broker':
        node.update({'broker': HOST, 'port': str(PORT), 'tls': True,
                     'name': 'HiveMQ Cloud',
                     'credentials': {'user': USER, 'password': PASS}})
        print('  ✅ MQTT broker → ' + HOST + ':' + str(PORT) + ' (TLS)')
    if node.get('type') == 'http request' and 'analyze' in node.get('url',''):
        node['url'] = PUBLIC_URL + '/analyze'
        print('  ✅ HTTP URL → ' + node['url'])

out = '/content/pump-iot-demo/nodered/flow_cloud.json'
with open(out, 'w') as f:
    json.dump(flow, f, indent=2)
print('\n✅ flow_cloud.json sẵn sàng: ' + out)

### Import vào FlowFuse

1. Trong **Colab Files panel** — tìm `flow_cloud.json` → chuột phải → **Download**
2. Mở **FlowFuse** → Instance → **Open Editor**
3. Trong Node-RED Editor: ☰ Menu → **Import** → chọn file vừa download
4. Nhấn **Deploy** *(nút đỏ)*

> ⚠️ Flow sẽ báo lỗi HTTP endpoint — bình thường vì URL là placeholder.
> Sẽ cập nhật lại sau khi có Cloudflare URL ở **Phần 6**.

---
## 💻 FALLBACK — Khởi động Node-RED local *(Bỏ qua nếu dùng FlowFuse)*

In [ ]:
# ── 💻 FALLBACK ONLY ─────────────────────────────────────────
import subprocess, time, shutil, os
NR_HOME = '/root/.node-red'
os.makedirs(NR_HOME, exist_ok=True)

flow_src = '/content/pump-iot-demo/nodered/flows.json'
if not os.path.exists(flow_src):
    print('❌ flows.json chưa có — upload theo hướng dẫn trên')
else:
    # Copy flows.json vào thư mục hệ thống của Node-RED
    shutil.copy(flow_src, os.path.join(NR_HOME, 'flows.json'))
    with open(os.path.join(NR_HOME, 'settings.js'), 'w') as f:
        f.write('module.exports={uiPort:1880,httpAdminRoot:"/",userDir:"/root/.node-red",flowFile:"flows.json"};')
    nr_proc = subprocess.Popen(
        ['node-red', '--settings', NR_HOME + '/settings.js'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=NR_HOME)
    print('⏳ Khởi động Node-RED...')
    for _ in range(60):
        time.sleep(1)
        line = nr_proc.stdout.readline().decode('utf-8', errors='replace')
        if 'Started flows' in line or 'running' in line.lower():
            print(f'✅ Node-RED chạy (PID {nr_proc.pid}) | localhost:1880')
            break
    else:
        print('⚠️ Node-RED đang khởi động...')

---
# Phần 6 — Dashboard & FastAPI Server

## Dashboard gồm 2 file

| File | Mô tả |
|------|-------|
| `index.html` | Dashboard chính: 4 tab (Live Feed · Timeline · AI Recommendation · Sensor Status) |
| `control.html` | Control Panel: nút simulate Normal / Warning / Critical |

Dashboard kết nối backend qua **WebSocket** để nhận dữ liệu real-time:
```javascript
const ws = new WebSocket(`wss://${location.host}/ws`);
ws.onmessage = e => {
    const data = JSON.parse(e.data);
    updateCharts(data);     // Chart.js timeline
    updateGauges(data);     // SVG arc gauges
    updateSensorPage(data); // Sensor status grid
}
```

### 📋 Thực hành: Upload `index.html`

*Dashboard chính với 4 tab, real-time charts và SVG gauges.*

1. Mở **Files panel** bên trái *(icon 📁 hoặc `Ctrl+Shift+F`)*
2. Điều hướng đến thư mục: **`/content/pump-iot-demo/dashboard/`**
3. Kéo thả file **`index.html`** vào &nbsp;*hoặc* chuột phải → **Upload**

| &nbsp; | Đường dẫn |
|--------|-----------|
| 📦 Trong ZIP | `pump-iot-demo/dashboard/index.html` |
| 📍 Upload đến | `/content/pump-iot-demo/dashboard/index.html` |

> Sau khi upload xong, chạy cell bên dưới để xác nhận.

In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/index.html'
assert os.path.exists(_p), '❌ Chưa thấy index.html — hãy upload lại'
_kb = max(1, os.path.getsize(_p) // 1024)
print('✅ index.html (' + str(_kb) + ' KB) — đã sẵn sàng!')
print('\n--- 6 dòng đầu ---')
with open(_p, encoding='utf-8', errors='replace') as _f:
    for _i, _l in enumerate(_f):
        if _i >= 6: print('  ...'); break
        print(f'  {_i+1:3d}  {_l.rstrip()}')

### 📋 Thực hành: Upload `control.html`

*Trang điều khiển — dùng để simulate các kịch bản sự cố trong demo.*

1. Mở **Files panel** bên trái *(icon 📁 hoặc `Ctrl+Shift+F`)*
2. Điều hướng đến thư mục: **`/content/pump-iot-demo/dashboard/`**
3. Kéo thả file **`control.html`** vào &nbsp;*hoặc* chuột phải → **Upload**

| &nbsp; | Đường dẫn |
|--------|-----------|
| 📦 Trong ZIP | `pump-iot-demo/dashboard/control.html` |
| 📍 Upload đến | `/content/pump-iot-demo/dashboard/control.html` |

> Sau khi upload xong, chạy cell bên dưới để xác nhận.

In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/control.html'
assert os.path.exists(_p), '❌ Chưa thấy control.html — hãy upload lại'
_kb = max(1, os.path.getsize(_p) // 1024)
print('✅ control.html (' + str(_kb) + ' KB) — đã sẵn sàng!')
print('\n--- 6 dòng đầu ---')
with open(_p, encoding='utf-8', errors='replace') as _f:
    for _i, _l in enumerate(_f):
        if _i >= 6: print('  ...'); break
        print(f'  {_i+1:3d}  {_l.rstrip()}')

## 6.1 Khởi động FastAPI server

Sau khi có `server.py` (Phần 3) và 2 file dashboard, ta khởi động backend:

In [ ]:
import subprocess, time
fastapi_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/pump-iot-demo/backend',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print('⏳ Đợi FastAPI...')
for _ in range(20):
    time.sleep(1)
    line = fastapi_proc.stdout.readline().decode('utf-8', errors='replace')
    if 'Application startup complete' in line or 'Uvicorn running' in line:
        print(f'✅ FastAPI chạy (PID {fastapi_proc.pid}) | localhost:8000')
        break
else:
    if fastapi_proc.poll() is None:
        print('⚠️ Đang khởi động... Chạy cell health check bên dưới')
    else:
        print('❌ Thoát sớm — đọc log:')
        print(fastapi_proc.stdout.read(2000).decode(errors='replace'))

In [ ]:
import requests, time
time.sleep(2)
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print('✅ Health check: ' + str(r.status_code))
    print('   ' + str(r.json()))
except Exception as e:
    print('❌ ' + str(e))

## 6.2 Tạo Public URL (Cloudflare Tunnel)

Colab không có IP công khai. Cloudflare Tunnel tạo URL HTTPS miễn phí để:
- Mở dashboard trên trình duyệt
- FlowFuse gửi dữ liệu về backend

> ⏱️ URL có hiệu lực ~2 giờ. Nếu hết hạn, chạy lại cell này.

In [ ]:
from pycloudflared import try_cloudflare
import threading
PUBLIC_URL = None

def _t():
    global PUBLIC_URL
    result = try_cloudflare(port=8000)
    # try_cloudflare trả về Urls namedtuple — lấy .tunnel để có string
    PUBLIC_URL = result.tunnel if hasattr(result, 'tunnel') else str(result)
    print('\n' + '='*60)
    print('  🌐 DASHBOARD : ' + PUBLIC_URL + '/dashboard/')
    print('  🎮 CONTROLS  : ' + PUBLIC_URL + '/control/')
    print('  📖 API DOCS  : ' + PUBLIC_URL + '/docs')
    print('='*60)

t = threading.Thread(target=_t, daemon=True)
t.start(); t.join(timeout=20)
if not PUBLIC_URL: print('⚠️ Thử chạy lại cell này')

## 6.3 ☁️ Cập nhật FlowFuse với URL thật

Tái tạo `flow_final.json` với Cloudflare URL đúng, rồi re-import vào FlowFuse:

In [ ]:
import json, os
flow_src = '/content/pump-iot-demo/nodered/flows.json'
if not os.path.exists(flow_src):
    print('⚠️ flows.json chưa có — upload ở Phần 5 trước')
elif not PUBLIC_URL:
    print('⚠️ PUBLIC_URL chưa có — chạy lại cell 6.2')
else:
    from dotenv import load_dotenv
    load_dotenv('/content/pump-iot-demo/backend/.env')
    HOST = os.environ.get('MQTT_HOST', 'localhost')
    PORT = int(os.environ.get('MQTT_PORT', 8883))
    USER = os.environ.get('MQTT_USERNAME', '')
    PASS = os.environ.get('MQTT_PASSWORD', '')
    # Đảm bảo PUBLIC_URL là string thuần
    _url = PUBLIC_URL.tunnel if hasattr(PUBLIC_URL, 'tunnel') else str(PUBLIC_URL)
    with open(flow_src) as f: flow = json.load(f)
    nodes = flow if isinstance(flow,list) else flow.get('flows',[])
    for node in nodes:
        if node.get('type') == 'mqtt-broker':
            node.update({'broker':HOST,'port':str(PORT),'tls':True,
                         'credentials':{'user':USER,'password':PASS}})
        if node.get('type')=='http request' and 'analyze' in node.get('url',''):
            node['url'] = _url + '/analyze'
            print('  ✅ HTTP URL → ' + node['url'])
    out = '/content/pump-iot-demo/nodered/flow_final.json'
    with open(out,'w') as f: json.dump(flow,f,indent=2)
    print('\n✅ flow_final.json sẵn sàng')
    print('  1. Download từ Colab Files panel')
    print('  2. FlowFuse Editor → ☰ → Import → chọn file')
    print('  3. Deploy')
    print('\n🌐 Dashboard: ' + _url + '/dashboard/')


---
# Phần 7 — Sensor Simulator

## simulate_sensor.py là gì?

Script Python giả lập **52 cảm biến** trên máy bơm công nghiệp, publish qua MQTT mỗi 0.5s:

| Nhóm | Sensor IDs | Đơn vị | Ngưỡng warn / crit |
|------|------------|--------|---------------------|
| ⚡ Vibration | sensor_00 – sensor_21 (22) | mm/s | 4.5 / 6.5 |
| 🌡 Temperature | sensor_22 – sensor_31 (10) | °C | 85 / 95 |
| 📊 Pressure | sensor_32 – sensor_41 (10) | bar | 8 / 10 |
| 💧 Flow Rate | sensor_42 – sensor_51 (10) | m³/h | < 120 / < 100 |

Hỗ trợ 3 mode: `normal` · `warning` · `critical`

### 📋 Thực hành: Upload `simulate_sensor.py`

*Script giả lập sensor — cần có trước khi chạy simulator.*

1. Mở **Files panel** bên trái *(icon 📁 hoặc `Ctrl+Shift+F`)*
2. Điều hướng đến thư mục: **`/content/pump-iot-demo/scripts/`**
3. Kéo thả file **`simulate_sensor.py`** vào &nbsp;*hoặc* chuột phải → **Upload**

| &nbsp; | Đường dẫn |
|--------|-----------|
| 📦 Trong ZIP | `pump-iot-demo/scripts/simulate_sensor.py` |
| 📍 Upload đến | `/content/pump-iot-demo/scripts/simulate_sensor.py` |

> Sau khi upload xong, chạy cell bên dưới để xác nhận.

In [ ]:
import os
_p = '/content/pump-iot-demo/scripts/simulate_sensor.py'
assert os.path.exists(_p), '❌ Chưa thấy simulate_sensor.py — hãy upload lại'
_kb = max(1, os.path.getsize(_p) // 1024)
print('✅ simulate_sensor.py (' + str(_kb) + ' KB) — đã sẵn sàng!')
print('\n--- 12 dòng đầu ---')
with open(_p, encoding='utf-8', errors='replace') as _f:
    for _i, _l in enumerate(_f):
        if _i >= 12: print('  ...'); break
        print(f'  {_i+1:3d}  {_l.rstrip()}')

## 7.1 Khởi động simulator — Normal mode

Sau khi upload, chạy simulator. Mở **Dashboard URL** để thấy dữ liệu real-time.

In [ ]:
import subprocess, time, os
path = '/content/pump-iot-demo/scripts/simulate_sensor.py'
if not os.path.exists(path):
    print('❌ simulate_sensor.py chưa có — upload theo hướng dẫn trên')
else:
    sim_proc = subprocess.Popen(
        ['python', 'simulate_sensor.py', '--mode', 'normal', '--interval', '0.5'],
        cwd='/content/pump-iot-demo/scripts',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(3)
    if sim_proc.poll() is None:
        print(f'✅ Simulator chạy (PID {sim_proc.pid})')
        print('   Mode    : normal')
        print('   Interval: 0.5s (2 msg/giây)')
        print('   Topic   : pump/sensors')
        if 'PUBLIC_URL' in dir() and PUBLIC_URL:
            print('\n💡 Dashboard: ' + PUBLIC_URL + '/dashboard/')
    else:
        log = sim_proc.stdout.read(1000).decode(errors='replace')
        print('❌ Lỗi:\n' + log)

---
# Phần 8 — Kiểm tra Pipeline End-to-End

Toàn bộ pipeline đang chạy:
```
[simulate_sensor.py]
  └─ MQTT → [HiveMQ Cloud] → [FlowFuse Node-RED]
                                     │ POST /sensor-update
                             [FastAPI :8000]
                                     │ WebSocket
                             [Dashboard Browser]
```

In [ ]:
import requests
print('=== Pipeline Status ===')
for name, url in [('FastAPI health','http://localhost:8000/health'),
                  ('Latest data',   'http://localhost:8000/latest')]:
    try:
        r = requests.get(url, timeout=3)
        d = r.json()
        has_data = any(k in str(d) for k in ['vibration','temperature','status'])
        label = 'dữ liệu đang chảy' if has_data else str(d)[:60]
        print('  ✅ ' + name + ': ' + label)
    except Exception as e:
        print('  ⚠️ ' + name + ': ' + str(e))
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('\n🌐 ' + PUBLIC_URL + '/dashboard/')

---
# Phần 9 — AI Analysis & Email Alerts

## Luồng phân tích

```
Node-RED (anomaly_score ≥ 0.4)
    └─ POST /alert (throttle 60s)
           └─ AI API → JSON response → WebSocket → Dashboard
           └─ Resend → Email HTML → Inbox
```

In [ ]:
import requests, json
test = {
    'machine_id': 'PUMP-001', 'status': 'warning',
    'vibration':   {'avg': 5.2,  'trend': 0.8,  'anomaly_score': 0.72},
    'temperature': {'avg': 88.5, 'trend': 0.3,  'anomaly_score': 0.65},
    'pressure':    {'avg': 8.9,  'trend': -0.1, 'anomaly_score': 0.45},
    'flow_rate':   {'avg': 135.2,'trend': -2.1, 'anomaly_score': 0.38},
}
print('📤 Test AI analysis...')
try:
    r = requests.post('http://localhost:8000/alert', json=test, timeout=30)
    result = r.json()
    if result.get('status') == 'skipped':
        print('  ℹ️ ' + result.get('reason','') + ' (mock data vẫn hiện trên dashboard)')
    else:
        ai = result.get('ai_analysis', {})
        print('  ✅ Risk: ' + str(ai.get('risk_level')))
        for rec in ai.get('recommendations', [])[:2]:
            print('  → ' + str(rec))
except Exception as e:
    print('❌ ' + str(e))

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')
key = os.environ.get('RESEND_API_KEY', '')
if key and len(key) > 10:
    print('✅ Resend đã cấu hình')
    print('   From : ' + os.environ.get('ALERT_FROM',''))
    print('   To   : ' + os.environ.get('ALERT_TO',''))
    print('   Email gửi khi WARNING/CRITICAL (cooldown 60s)')
else:
    print('ℹ️ Resend chưa cấu hình — email sẽ bị bỏ qua')
    print('   Để kích hoạt: đăng ký resend.com → cập nhật RESEND_API_KEY trong .env')

---
# Phần 10 — Demo: Mô phỏng Sự cố

| Kịch bản | Triệu chứng | Nguyên nhân thực tế |
|---------|-------------|---------------------|
| 🔴 Bearing Wear | Vibration tăng dần, temp tăng | Ổ trục mài mòn |
| 🔴 Cavitation | Pressure dao động, flow giảm | Áp suất hút thấp |
| 🔴 Overheating | Temperature tăng nhanh | Thiếu bôi trơn |

In [ ]:
import subprocess, time, requests
print('🔴 Chuyển sang CRITICAL...')
try:
    r = requests.post('http://localhost:8000/simulate', json={'mode':'critical'}, timeout=5)
    print('   ✅ ' + str(r.json()))
except:
    if 'sim_proc' in dir() and sim_proc.poll() is None: sim_proc.terminate()
    sim_proc = subprocess.Popen(
        ['python','simulate_sensor.py','--mode','critical','--interval','0.5'],
        cwd='/content/pump-iot-demo/scripts',
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('   ✅ Simulator critical PID ' + str(sim_proc.pid))
print('\n💡 Quan sát dashboard:')
print('   → Sensors đỏ')
print('   → AI tab tự phân tích sau ~10s')
if 'PUBLIC_URL' in dir() and PUBLIC_URL: print('   → ' + PUBLIC_URL + '/dashboard/')

In [ ]:
# Reset về Normal
if 'sim_proc' in dir() and sim_proc.poll() is None: sim_proc.terminate()
import subprocess
sim_proc = subprocess.Popen(
    ['python','simulate_sensor.py','--mode','normal','--interval','0.5'],
    cwd='/content/pump-iot-demo/scripts',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✅ Reset về Normal (PID ' + str(sim_proc.pid) + ')')

---
# Phần 11 — Giám sát & Troubleshooting

In [ ]:
import requests
def svc(name, proc, url=None):
    alive = proc is not None and proc.poll() is None
    icon = '🟢' if alive else '🔴'
    out = icon + ' ' + name
    if alive and url:
        try:
            r = requests.get(url, timeout=2)
            out += ' | HTTP ' + str(r.status_code)
        except: out += ' | unreachable'
    print(out)

print('═'*46)
svc('FastAPI :8000',   locals().get('fastapi_proc'), 'http://localhost:8000/health')
svc('Simulator',       locals().get('sim_proc'))
for n,v in [('Mosquitto','mosquitto_proc'),('Node-RED','nr_proc')]:
    if v in dir(): svc(n, eval(v))
print('═'*46)
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('\n🌐 ' + PUBLIC_URL + '/dashboard/')

| Triệu chứng | Nguyên nhân | Cách xử lý |
|-------------|-------------|------------|
| Dashboard trắng | FastAPI chưa chạy | Chạy lại Phần 6.1 |
| Không nhận dữ liệu | FlowFuse chưa deploy | Re-deploy flow |
| AI tab xoay mãi | Không có API key | Thêm key → restart backend |
| Email không gửi | RESEND_KEY sai | Check resend.com Logs |
| MQTT lỗi | Sai credentials HiveMQ | Kiểm tra console.hivemq.cloud |
| URL hết hạn | Cloudflare ~2h | Chạy lại 6.2 + cập nhật FlowFuse |
| Colab reset | Idle > 90 phút | Chạy lại từ Phần 6 (files vẫn còn) |

---
# Phần 12 — Deploy lên AWS

## Kiến trúc EC2 (Option A — Single Instance)

```
Internet ──HTTPS──► Nginx (443)
         ──MQTTS──► Mosquitto (8883)
                    ┌─────────────────────────────────┐
                    │  EC2 t3.small (Ubuntu 22.04)     │
                    │  FastAPI :8000  Node-RED :1880   │
                    │  Mosquitto :1883  Simulator       │
                    └─────────────────────────────────┘
```

**Security Group — ports cần mở:**

| Port | Nguồn | Dùng cho |
|------|-------|----------|
| 22 | Your IP | SSH |
| 80 / 443 | 0.0.0.0/0 | HTTP/HTTPS (Nginx) |
| 1883 / 8883 | 0.0.0.0/0 | MQTT / MQTTS |
| 1880 | Your IP | Node-RED Editor |

In [ ]:
import os
aws_dir = '/content/pump-iot-demo/scripts/aws'
os.makedirs(aws_dir, exist_ok=True)

setup = '''#!/bin/bash
# PumpGuard AI — EC2 Ubuntu 22.04 Setup
set -e
sudo apt-get update -y && sudo apt-get upgrade -y
sudo apt-get install -y python3.11 python3.11-pip python3.11-venv
sudo apt-get install -y mosquitto mosquitto-clients nginx certbot python3-certbot-nginx
sudo systemctl enable mosquitto
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs && sudo npm install -g node-red
cd ~/pump-iot-demo/backend
python3.11 -m venv .venv && source .venv/bin/activate
pip install fastapi 'uvicorn[standard]' paho-mqtt python-dotenv anthropic openai google-generativeai resend
echo Done
'''

backend_svc = '''[Unit]
Description=PumpGuard FastAPI
After=network.target mosquitto.service
[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/pump-iot-demo/backend
ExecStart=/home/ubuntu/pump-iot-demo/backend/.venv/bin/uvicorn server:app --host 0.0.0.0 --port 8000
Restart=always
[Install]
WantedBy=multi-user.target
'''

nr_svc = '''[Unit]
Description=PumpGuard Node-RED
After=network.target mosquitto.service
[Service]
User=ubuntu
ExecStart=/usr/bin/node-red --userDir /home/ubuntu/.node-red
Restart=always
[Install]
WantedBy=multi-user.target
'''

sim_svc = '''[Unit]
Description=PumpGuard Simulator
After=network.target mosquitto.service
[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/pump-iot-demo/scripts
ExecStart=/usr/bin/python3 simulate_sensor.py --mode normal --interval 0.5
Restart=always
[Install]
WantedBy=multi-user.target
'''

files = {'setup_ec2.sh': setup,
         'pumpguard-backend.service': backend_svc,
         'pumpguard-nodered.service': nr_svc,
         'pumpguard-simulator.service': sim_svc}
for name, content in files.items():
    path = os.path.join(aws_dir, name)
    with open(path,'w') as f: f.write(content)
    if name.endswith('.sh'): os.chmod(path, 0o755)
    print('✅ ' + name)
print('\nDeploy: scp -r ' + aws_dir + '/ ubuntu@EC2_IP:~/')
print('Rồi SSH vào EC2: bash setup_ec2.sh')

---
# Phần 13 — Tổng kết & Bài tập

## 🎉 Hệ thống đã hoàn chỉnh!

| Bạn đã xây dựng | Tương đương production |
|----------------|----------------------|
| HiveMQ Cloud | AWS IoT Core / Azure IoT Hub |
| FlowFuse Node-RED | Industrial Edge Gateway |
| FastAPI + WebSocket | Real-time telemetry platform |
| AI analysis | Predictive maintenance ML |
| Resend email | PagerDuty / OpsGenie |
| EC2 + systemd | Production on-premise server |

---

## 🚀 Bài tập mở rộng

**Cơ bản:**
1. Thêm loại sensor `acoustic_emission` (âm thanh siêu âm)
2. Điều chỉnh ngưỡng cảnh báo theo datasheet thực tế của máy bơm
3. Thêm Tiếng Việt vào AI system prompt

**Trung bình:**
4. Lưu lịch sử vào SQLite + endpoint `/history?hours=24`
5. Thêm Basic Auth cho Control Panel
6. Tích hợp Grafana để visualize từ SQLite

**Nâng cao:**
7. Thay rule-based bằng **Isolation Forest** (scikit-learn)
8. Gắn cảm biến MPU-6050 thực lên Raspberry Pi
9. Migrate lên **AWS IoT Core + Lambda + DynamoDB**

---

**Tài liệu:** [HiveMQ](https://docs.hivemq.com/) · [FlowFuse](https://flowfuse.com/docs/) · [FastAPI](https://fastapi.tiangolo.com/) · [AWS IoT](https://docs.aws.amazon.com/iot/)